# Hypothesis 3 — T4 采集与评价
流程为 **配置 → 跑题 → 评价 → 下载**。

固定题库 11,569 个请求/模型、12,767 个 typed decisions。
CPU 生产者预处理、GPU 消费者自适应合批；模型依次加载。
使用 Colab 自带 Python / PyTorch，不访问 Google Drive、不需要 GH_TOKEN、不做 smoke。

评价只用 CPU，不重新加载模型。单选主指标为去除随机基线分：
100 × (准确率 − 1/K) / (1 − 1/K)，同时保留原始准确率和聚类置信区间。
多选另报整题全对率、集合 F1、全选率、随机与全选基线，不与单选混合排名。

当前采集适配器仍仅支持 Laya English / Multilingual。
既有研究共 23 个模型，Jev API 跳过；其他尚无 H3 结果的模型在覆盖表标为 NOT_RUN。
评价器可读取所有适配到相同 H3 输出格式的模型，并不局限于 Laya。

本评价方法在已查看串行 Laya 数据后制定，该批属于探索性结果。
题库语义复核、协作者最终 H1/H2 排除清单仍待完成。


In [ ]:
# 1. 配置：安装依赖、准备本实验题库（无需模型推理）
import sys, json, subprocess, hashlib
from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone
from uuid import uuid4

MODELS = ["english", "multilingual"]
SCHEDULER = "adaptive"  # 可改成 serial，使用逐题推理
ADAPTIVE = {
    "initial_batch_rows": 8, "max_batch_rows": 128, "max_padded_tokens": 32768,
    "producer_workers": 2, "prefetch_cases": 64, "lookahead_cases": 64,
    "sample_interval": 2.0, "low_samples": 16, "grow_below": 0.70,
    "shrink_above": 0.85, "reserve_gib": 2.0,
}
BOOTSTRAP_REPS = 2000  # CPU 聚类重采样次数；可改为 10000
EVALUATION_SEED = 20261026
PROFILE = "full"  # 完整题库；可手动改为 pilot，只采集60道开发母题的所有配对变体
EXCLUSION_FILES = []  # 若拿到 H1/H2 最终排除清单，填入 JSON 文件路径
assert MODELS and all(m in ("english", "multilingual") for m in MODELS)
assert PROFILE in ("full", "pilot")
assert Path("/content").exists(), "请在 Colab 中执行"
assert sys.version_info >= (3, 12), "需要 Python 3.12+"

LOG_DIR = Path("/content/hypothesis3-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_NUMBER = 0

def run(args, cwd=None):
    global COMMAND_NUMBER
    args = list(map(str, args))
    COMMAND_NUMBER += 1
    log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    while log.exists():
        COMMAND_NUMBER += 1
        log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    print("+", " ".join(args), "\n日志:", log, flush=True)
    tail = []
    with log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(
            args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
                tail.append(line)
                tail = tail[-40:]
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if code:
        raise RuntimeError(f"退出码 {code}；日志 {log}\n{''.join(tail)}")

run(["nvidia-smi"])
import torch
assert torch.cuda.is_available(), "请选择 T4 GPU 运行时"
TORCH_VERSION = version("torch")
print("复用 PyTorch:", TORCH_VERSION, "GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/system1bench")
if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/Jaffe2718/system1bench.git", REPO])
    run(["git", "sparse-checkout", "set", "system1bench", "research/hypothesis3_v1"], cwd=REPO)
else:
    run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO)
    if (REPO / ".git/info/sparse-checkout").exists():
        run(["git", "sparse-checkout", "add", "research/hypothesis3_v1"], cwd=REPO)

PY = Path(sys.executable)
EXPERIMENT = REPO / "research/hypothesis3_v1"
for required in ("run.py", "evaluate.py", "model_cohort.json"):
    assert (EXPERIMENT / required).is_file(), required
CONSTRAINT = Path("/content/hypothesis3-torch-constraint.txt")
CONSTRAINT.write_text("torch==" + TORCH_VERSION)
run([PY, "-m", "pip", "install", "--disable-pip-version-check",
     "--constraint", CONSTRAINT, "-r", "requirements-colab-t4.txt",
     "-r", EXPERIMENT / "requirements-runtime.txt"], cwd=REPO)
run([PY, "-c",
     "import sys, torch, numpy, pyarrow.parquet, laya, psutil; "
     "from importlib.metadata import version; "
     "from laya.common import build_sequence, render_options, serialize_state; "
     "assert version('torch') == sys.argv[1]; print('依赖导入成功，PyTorch 保持不变')",
     TORCH_VERSION], cwd=REPO)

BANK_REVISION = hashlib.sha256(
    (EXPERIMENT / "build.py").read_bytes() + (EXPERIMENT / "sources.json").read_bytes()).hexdigest()[:12]
BANK = Path("/content/hypothesis3-banks") / BANK_REVISION
CACHE = Path("/content/hypothesis3-cache/raw")
# 外部排除清单也进入目录标识，避免复用未经该清单过滤的旧题库。
if EXCLUSION_FILES:
    extra = hashlib.sha256(b"".join(Path(p).read_bytes() for p in EXCLUSION_FILES)).hexdigest()[:12]
    BANK = BANK.with_name(BANK.name + "-" + extra)
if BANK.exists():
    run([PY, EXPERIMENT / "build.py", "--output", BANK, "--verify"], cwd=REPO)
else:
    args = [PY, "-u", EXPERIMENT / "build.py", "--cache", CACHE, "--output", BANK]
    for filename in EXCLUSION_FILES:
        args += ["--exclude", filename]
    run(args, cwd=REPO)

# 默认题库还要与已冻结的公开构建回执逐文件比对。
if not EXCLUSION_FILES:
    receipt = json.loads((EXPERIMENT / "bank_receipt.json").read_text())
    assert hashlib.sha256((BANK / "manifest.json").read_bytes()).hexdigest() == receipt["manifest_sha256"]
    for filename, expected in receipt["files"].items():
        assert hashlib.sha256((BANK / filename).read_bytes()).hexdigest() == expected["sha256"], filename
    print("题库与冻结回执完全一致。")

SESSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUTPUT = Path("/content/hypothesis3-results") / SESSION
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "configuration.json").write_text(json.dumps(
    {"models":MODELS,"profile":PROFILE,"bank":str(BANK),
     "scheduler":SCHEDULER,"adaptive":ADAPTIVE,
     "evaluation_planned":True,"bootstrap_repetitions":BOOTSTRAP_REPS,
     "evaluation_seed":EVALUATION_SEED},
    ensure_ascii=False,indent=2))
print("配置完成。下一步加载模型后直接跑本实验题库。", BANK, OUTPUT)


In [ ]:
# 2. 跑题：模型下载/加载完成后立即采集，不做 smoke，不进行研究评价
scheduler_args = ["--scheduler", SCHEDULER]
for key, value in ADAPTIVE.items():
    scheduler_args += ["--" + key.replace("_", "-"), str(value)]
run([PY, "-u", EXPERIMENT / "run.py", "--bank", BANK, "--output", OUTPUT,
     "--models", *MODELS, "--profile", PROFILE, *scheduler_args], cwd=REPO)

status = json.loads((OUTPUT / "run_status.json").read_text())
print("运行状态（不是模型分数）:", status["status"])
for model, result in status["models"].items():
    print(model, result["status"], "已处理请求:", result.get("completed_requests", 0),
          "运行失败请求:", result.get("failed_requests", 0))
print("采集结束；下一格在 CPU 上评价已有结果，不重新推理。")


In [ ]:
# 3. 评价：CPU 读取预测，校验后生成分数、区间和全模型覆盖表
EVALUATION = OUTPUT / "evaluation"
run([PY, "-u", EXPERIMENT / "evaluate.py",
     "--results", OUTPUT, "--bank", BANK, "--output", EVALUATION,
     "--bootstrap", BOOTSTRAP_REPS, "--seed", EVALUATION_SEED], cwd=REPO)

evaluation_status = json.loads((EVALUATION / "status.json").read_text(encoding="utf-8"))
print("已评价模型:", evaluation_status["models_evaluated"],
      "未运行:", evaluation_status["models_not_run"],
      "失败或无法评价:", evaluation_status["models_unevaluable"])
from IPython.display import display, Markdown
display(Markdown((EVALUATION / "REPORT.md").read_text(encoding="utf-8")))


In [ ]:
# 3. 下载：原始预测 + 题库/参考答案 + 版本/代码/日志，不包含研究评分
import shutil
assert (OUTPUT / "evaluation/summary.json").is_file(), "请先运行第 3 步评价"
from google.colab import files

(OUTPUT / "environment-pip-freeze.txt").write_text(subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True))
(OUTPUT / "repository-commit.txt").write_text(subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())
(OUTPUT / "working-tree.diff").write_text(subprocess.check_output(
    ["git", "diff", "--no-ext-diff", "HEAD"], cwd=REPO, text=True))
shutil.copytree(BANK, OUTPUT / "question_bank", dirs_exist_ok=True)
shutil.copytree(LOG_DIR, OUTPUT / "configuration-logs", dirs_exist_ok=True)
code = OUTPUT / "collection_code"
code.mkdir(exist_ok=True)
for name in ("build.py", "run.py", "adaptive.py", "evaluate.py", "model_cohort.json", "EVALUATION.md", "requirements-runtime.txt", "sources.json", "bank_receipt.json", "PROTOCOL.md", "README.md", "NOTICE.md"):
    shutil.copyfile(EXPERIMENT / name, code / name)
archive = shutil.make_archive("/content/hypothesis3-t4-results", "zip", OUTPUT)
print("ZIP 包含原始预测及 evaluation/ 下的分数、报告、统计协议和模型覆盖表。")
files.download(archive)
